#**Collaborative Filtering**

In [1]:
# Import PySpark components and auxiliary libraries for data manipulation and analysis
from google.colab import drive

import os
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count

from pyspark.sql import functions as F

from pyspark.sql.functions import collect_list, sort_array, udf
from pyspark.sql.types import ArrayType, LongType

from pyspark.sql.functions import explode, col, array, lit

In [2]:
# Mount Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Find Project Folder
def find_project_folder(folder_name="CineFusion", base_path="/content/drive/MyDrive"):
    for root, dirs, files in os.walk(base_path):
        if folder_name in dirs:
            return os.path.join(root, folder_name)
    return None

BASE_DIR = find_project_folder()

if BASE_DIR is None:
    raise Exception("Project folder not found. Make sure it's added to MyDrive.")

print("Project folder found at:", BASE_DIR)

Project folder found at: /content/drive/MyDrive/CineFusion


In [4]:
# Start Spark Session
spark = SparkSession.builder \
    .appName("CineFusion - CF") \
    .config("spark.driver.memory", "12g") \
    .config("spark.executor.memory", "12g") \
    .getOrCreate()

In [5]:
# Import Rating Data from the 25M Dataset
ratings = spark.read.csv(
    f"{BASE_DIR}/ml-25m/ratings.csv",
    header=True,
    inferSchema=True
).select("userId", "movieId", "rating")

movies = spark.read.csv(
    f"{BASE_DIR}/ml-25m/movies.csv",
    header=True,
    inferSchema=True
).select("movieId", "title")

print("Ratings count:", ratings.count())
print("Movies count:", movies.count())

Ratings count: 25000095
Movies count: 62423


In [6]:
# Counting unique entries
ratings_stats = ratings.agg(
    F.min("userId").alias("min_userId"),
    F.max("userId").alias("max_userId"),
    F.min("movieId").alias("min_movieId"),
    F.max("movieId").alias("max_movieId"),
    F.countDistinct("userId").alias("unique_userId"),
    F.countDistinct("movieId").alias("unique_movieId")
).collect()[0]

movies_stats = movies.agg(
    F.min("movieId").alias("min_movieId"),
    F.max("movieId").alias("max_movieId"),
    F.countDistinct("movieId").alias("unique_movieId")
).collect()[0]

print("rating.csv")
print(f"userId : {ratings_stats['min_userId']} -> {ratings_stats['max_userId']}")
print(f"movieId : {ratings_stats['min_movieId']} -> {ratings_stats['max_movieId']}")
print(f"unique userId : {ratings_stats['unique_userId']}")
print(f"unique movieId : {ratings_stats['unique_movieId']}")

print("\nmovies.csv")
print(f"movieId : {movies_stats['min_movieId']} -> {movies_stats['max_movieId']}")
print(f"unique movieId : {movies_stats['unique_movieId']}")

rating.csv
userId : 1 -> 162541
movieId : 1 -> 209171
unique userId : 162541
unique movieId : 59047

movies.csv
movieId : 1 -> 209171
unique movieId : 62423


In [7]:
# Group all movies rated by a user and sort them
user_movies = ratings.groupBy("userId") \
    .agg(sort_array(collect_list("movieId")).alias("movies"))

user_movies.show(5)

+------+--------------------+
|userId|              movies|
+------+--------------------+
|     1|[296, 306, 307, 6...|
|     3|[1, 29, 32, 50, 1...|
|     5|[1, 19, 32, 36, 3...|
|     6|[161, 260, 318, 5...|
|     9|[2, 10, 34, 39, 6...|
+------+--------------------+
only showing top 5 rows


In [ ]:
# Count total number of unique movies rated by all users
all_movies = user_movies.select(explode("movies").alias("movieId")).distinct()
all_movies_list = [row.movieId for row in all_movies.collect()]

print("Number of unique movies:", len(all_movies_list))

Number of unique movies: 59047


In [ ]:
# Generate 100 Permutations of all uniques movies arranged in a random order
k = 100
np.random.seed(42)
permutations = [np.random.permutation(all_movies_list).tolist()
                for _ in range(k)]
permutations_broadcast = spark.sparkContext.broadcast(permutations)

In [ ]:
def minhash_signature(movies):
    perms = permutations_broadcast.value
    sig = []
    movie_set = set(movies)
    for perm in perms:
        for idx, movie in enumerate(perm):
            if movie in movie_set:
                sig.append(idx)
                break
    return sig

minhash_udf = udf(minhash_signature, ArrayType(LongType()))

user_minhash = user_movies.withColumn(
    "minhash", minhash_udf("movies")
).select("userId", "minhash")

user_minhash.show(10)

+------+--------------------+
|userId|             minhash|
+------+--------------------+
|     1|[1666, 1185, 74, ...|
|     3|[30, 5, 74, 13, 1...|
|     5|[363, 44, 74, 430...|
|     6|[4150, 2137, 2034...|
|     9|[449, 616, 421, 2...|
|    12|[3, 241, 74, 13, ...|
|    13|[79, 71, 74, 13, ...|
|    15|[3, 14, 74, 1646,...|
|    16|[7658, 241, 4367,...|
|    17|[79, 334, 1114, 5...|
+------+--------------------+
only showing top 10 rows


In [ ]:
user_minhash.coalesce(1).write \
    .mode("append") \
    .parquet(f"{BASE_DIR}/ml-25m-output/minhash_parquet")

print(f"Saved {user_minhash.count()} user signatures to {BASE_DIR}/ml-25m-output/minhash_parquet")

Saved 162541 user signatures to /content/drive/MyDrive/CineFusion/ml-25m-output/minhash_parquet


In [ ]:
from pyspark.sql.functions import concat_ws

user_minhash_csv = user_minhash.withColumn(
    "minhash",
    concat_ws(",", "minhash")
)

user_minhash_csv.write \
    .mode("append") \
    .option("header", True) \
    .csv(f"{BASE_DIR}/ml-25m-output/minhash_csv")

print(f"Saved {user_minhash.count()} user signatures to {BASE_DIR}/ml-25m-output/minhash_csv")

Saved 162541 user signatures to /content/drive/MyDrive/CineFusion/ml-25m-output/minhash_csv


In [ ]:
from pyspark.sql.functions import col, slice, posexplode, concat_ws, collect_list

b = 20
r = 5
assert b * r == 100

In [ ]:
from pyspark.sql.functions import array

df = user_minhash

for i in range(b):
    df = df.withColumn(
        f"band_{i}",
        slice(col("minhash"), i * r + 1, r)
    )

# df.show()

In [ ]:
band_cols = [f"band_{i}" for i in range(b)]

df_bands = df.select(
    "userId",
    posexplode(array(*[col(c) for c in band_cols])).alias("band_id", "band_values")
)

# df_bands.show()

In [ ]:
from pyspark.sql.functions import hash

df_hashed = df_bands.withColumn(
    "bucket",
    hash(col("band_values"))
)

df_hashed.show()

+------+-------+--------------------+-----------+
|userId|band_id|         band_values|     bucket|
+------+-------+--------------------+-----------+
|     1|      0|[1666, 1185, 74, ...| -783911969|
|     1|      1|[223, 501, 459, 6...| 1292522331|
|     1|      2|[342, 89, 419, 20...|  145431253|
|     1|      3|[1034, 189, 560, ...|  981004731|
|     1|      4|[463, 906, 718, 1...| -914181784|
|     1|      5|[249, 206, 780, 2...|  161219859|
|     1|      6|[1804, 0, 174, 24...|-1551684307|
|     1|      7|[2793, 2945, 1725...| 1296143564|
|     1|      8|[84, 307, 1675, 6...| -786267448|
|     1|      9|[1093, 1166, 1796...|-2095862016|
|     1|     10|[331, 1654, 1340,...| 1608037185|
|     1|     11|[206, 338, 259, 1...| -157102998|
|     1|     12|[417, 1750, 455, ...| 1591976460|
|     1|     13|[969, 116, 137, 7...|-1587558520|
|     1|     14|[2447, 1219, 2834...|-1102942056|
|     1|     15|[263, 554, 1329, ...| -273309605|
|     1|     16|[34, 1516, 8, 215...|-1530362458|


In [ ]:
df_buckets = df_hashed.groupBy("band_id", "bucket") \
    .agg(collect_list("userId").alias("users")) \
    .filter("size(users) > 1")

df_buckets.show()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
from pyspark.sql.functions import explode

df_pairs = df_buckets.select(
    "band_id",
    explode("users").alias("user1"),
    "users"
).select(
    "band_id",
    "user1",
    explode("users").alias("user2")
).filter("user1 < user2")

In [ ]:
df_pairs.show()

In [ ]:
candidate_pairs = df_pairs.select("user1", "user2").distinct()

In [ ]:
print("Candidate pairs:", candidate_pairs.count())

In [ ]:
candidate_pairs.coalesce(1).write \
    .mode("append") \
    .parquet(f"{BASE_DIR}/ml-25m-output/candidate_pairs_parquet")

print(f"Saved {candidate_pairs.count()} candidate pairs to {BASE_DIR}/ml-25m-output/candidate_pairs_parquet")

In [ ]:
candidate_pairs.coalesce(1).write \
    .mode("append") \
    .option("header", True) \
    .csv(f"{BASE_DIR}/ml-25m-output/candidate_pairs_csv")

print(f"Saved {candidate_pairs.count()} candidate pairs to {BASE_DIR}/ml-25m-output/candidate_pairs_csv")

In [ ]:
candidate_pairs.show()

In [ ]:
# Join with user shingles
# pairs_with_data = candidate_pairs \
#     .join(user_shingles.withColumnRenamed("userId", "user1"), "user1") \
#     .join(user_shingles.withColumnRenamed("userId", "user2"), "user2")

# Then compute Jaccard (can show if you want)

# Clustering

In [8]:
candidate_pairs = spark.read.csv(
    f"{BASE_DIR}/ml-25m-output/candidate_pairs_csv",
    header=True,
    inferSchema=True
)

In [9]:
candidate_pairs.show(5)

+-----+------+
|user1| user2|
+-----+------+
|72352|106491|
|21096| 25775|
|80597|125514|
|21456|147877|
|21731| 61892|
+-----+------+
only showing top 5 rows


In [10]:
from pyspark.sql.functions import col

edges1 = candidate_pairs.select(
    col("user1").alias("user"),
    col("user2").alias("neighbor")
)

edges2 = candidate_pairs.select(
    col("user2").alias("user"),
    col("user1").alias("neighbor")
)

edges = edges1.union(edges2)

In [11]:
edges1.show(5)

+-----+--------+
| user|neighbor|
+-----+--------+
|72352|  106491|
|21096|   25775|
|80597|  125514|
|21456|  147877|
|21731|   61892|
+-----+--------+
only showing top 5 rows


In [12]:
edges2.show(5)

+------+--------+
|  user|neighbor|
+------+--------+
|106491|   72352|
| 25775|   21096|
|125514|   80597|
|147877|   21456|
| 61892|   21731|
+------+--------+
only showing top 5 rows


In [13]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, col

window = Window.partitionBy("user").orderBy(col("neighbor"))

limited_neighbors = edges.withColumn(
    "rank",
    row_number().over(window)
).filter(col("rank") <= 100)

In [14]:
limited_neighbors.show(5)

+----+--------+----+
|user|neighbor|rank|
+----+--------+----+
|   3|     598|   1|
|   3|    4310|   2|
|   3|    4758|   3|
|   3|    4802|   4|
|   3|    6836|   5|
+----+--------+----+
only showing top 5 rows


In [15]:
from pyspark.sql.functions import collect_list

user_neighbors = limited_neighbors.groupBy("user").agg(
    collect_list("neighbor").alias("neighbors")
)

In [16]:
user_neighbors.show(20, truncate=False)

+----+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user|neighbors                                                                                                                                                                                                                                                                        

to be contd

In [17]:
from pyspark.sql.functions import explode

pairs = user_neighbors.select(
    col("user"),
    explode("neighbors").alias("neighbor")
)

In [18]:
pairs.show(6)

+----+--------+
|user|neighbor|
+----+--------+
|   3|     598|
|   3|    4310|
|   3|    4758|
|   3|    4802|
|   3|    6836|
|   3|    7439|
+----+--------+
only showing top 6 rows


In [19]:
ratings1 = ratings.select(
    col("userId").alias("user"),
    col("movieId"),
    col("rating").alias("rating_user")
)

ratings2 = ratings.select(
    col("userId").alias("neighbor"),
    col("movieId"),
    col("rating").alias("rating_neighbor")
)

In [20]:
ratings1.show(5)

+----+-------+-----------+
|user|movieId|rating_user|
+----+-------+-----------+
|   1|    296|        5.0|
|   1|    306|        3.5|
|   1|    307|        5.0|
|   1|    665|        5.0|
|   1|    899|        3.5|
+----+-------+-----------+
only showing top 5 rows


In [21]:
ratings2.show(5)

+--------+-------+---------------+
|neighbor|movieId|rating_neighbor|
+--------+-------+---------------+
|       1|    296|            5.0|
|       1|    306|            3.5|
|       1|    307|            5.0|
|       1|    665|            5.0|
|       1|    899|            3.5|
+--------+-------+---------------+
only showing top 5 rows


In [22]:
joined = pairs \
    .join(ratings1, "user") \
    .join(ratings2, ["neighbor", "movieId"])

In [23]:
from pyspark.sql.functions import col

joined = joined.filter(
    (col("rating_user") > 0) & (col("rating_neighbor") > 0)
)

In [24]:
from pyspark.sql.functions import avg

means = joined.groupBy("user", "neighbor").agg(
    avg("rating_user").alias("mean_user"),
    avg("rating_neighbor").alias("mean_neighbor")
)

In [25]:
joined = joined.join(means, ["user", "neighbor"])

from pyspark.sql.functions import pow

centered = joined.withColumn(
    "ru_centered",
    col("rating_user") - col("mean_user")
).withColumn(
    "rv_centered",
    col("rating_neighbor") - col("mean_neighbor")
)

In [26]:
from pyspark.sql.functions import sum as spark_sum, sqrt

pearson = centered.groupBy("user", "neighbor").agg(
    spark_sum(col("ru_centered") * col("rv_centered")).alias("numerator"),
    sqrt(spark_sum(pow(col("ru_centered"), 2))).alias("denom_u"),
    sqrt(spark_sum(pow(col("rv_centered"), 2))).alias("denom_v")
)

In [27]:
from pyspark import StorageLevel
from pyspark.sql.functions import when

pearson = pearson.withColumn(
    "pearson",
    when(
        (col("denom_u") * col("denom_v")) != 0,
        col("numerator") / (col("denom_u") * col("denom_v"))
    ).otherwise(0)
)

pearson = pearson.persist(StorageLevel.MEMORY_AND_DISK)

In [28]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("user").orderBy(col("pearson").desc())

top_neighbors = pearson.withColumn(
    "rank",
    row_number().over(window)
)

In [29]:
top_neighbors = top_neighbors.filter(col("rank") <= 10)

In [ ]:
# top_neighbors.show(5)

+----+--------+------------------+-----------------+------------------+-------------------+----+
|user|neighbor|         numerator|          denom_u|           denom_v|            pearson|rank|
+----+--------+------------------+-----------------+------------------+-------------------+----+
|   3|   54573|38.321256038647306|8.053414674701989| 8.519939670013885| 0.5584999699247678|   1|
|   3|  103514| 49.63484848484851|7.368442128017251|13.824330769050665| 0.4872668959011266|   2|
|   3|   11052| 55.01815642458089|10.27312080982133|11.146936347899995|0.48044991091410016|   3|
|   3|   15539| 73.45639534883716|9.771666445201593|15.664264618759203|0.47990024070057924|   4|
|   3|    4802| 48.32429718875498|7.749983806175583| 13.09610751820993|0.47612668161263194|   5|
+----+--------+------------------+-----------------+------------------+-------------------+----+
only showing top 5 rows


In [30]:
from pyspark.sql.functions import collect_list

final_neighbors = top_neighbors.groupBy("user").agg(
    collect_list("neighbor").alias("neighbors")
)

In [31]:
final_neighbors.show(5)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

SAVE TOP(FINAL) NEIGHBOURS SORTED ON SIMILARITY AS PARQUET

In [ ]:

from pyspark import StorageLevel

final_neighbors = final_neighbors.persist(StorageLevel.MEMORY_AND_DISK)

final_neighbors.write \
    .mode("overwrite") \
    .parquet("/content/final_neighbors_parquet")

print("write done in parquet format")

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

SAVE TOP NEIGHBOURS SORTED ON SIMILARITY AS CSV

In [ ]:
from pyspark.sql.functions import concat_ws

final_neighbors_csv = final_neighbors.withColumn(
    "neighbors",
    concat_ws(",", "neighbors")
)

In [ ]:
final_neighbors_csv.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(f"{BASE_DIR}/ml-25m-output/final_neighbors_csv")

print("Saved CSV successfully ✅")

NameError: name 'final_neighbors_csv' is not defined

In [ ]:
final_neighbors.show(10, truncate=False)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [32]:
final_neighbors.take(3)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

Checking if sorting is correct


In [33]:
u = 148

In [34]:
neighbors = final_neighbors.filter(col("user") == u).collect()[0]["neighbors"]
print(neighbors)

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 720, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
pearson_u = pearson.filter(col("user") == u) \
    .select("neighbor", "pearson")

In [ ]:
pearson_top = pearson_u.filter(col("neighbor").isin(neighbors))

In [ ]:
pearson_sorted = pearson_top.orderBy(col("pearson").desc())

pearson_sorted.show(truncate=False)

# **CONTENT BASED FILTERING**

In [45]:
import pandas as pd
import ast

movies = pd.read_csv(f"{BASE_DIR}/tmdb-5000/tmdb_5000_movies.csv")
credits = pd.read_csv(f"{BASE_DIR}/tmdb-5000/tmdb_5000_credits.csv")

# merged the datasets
movies = movies.merge(credits, on="title")

In [46]:
movies.columns

Index(['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count', 'movie_id', 'cast', 'crew'],
      dtype='object')

In [47]:
movies["genres"][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [48]:
movies["keywords"][0]

'[{"id": 1463, "name": "culture clash"}, {"id": 2964, "name": "future"}, {"id": 3386, "name": "space war"}, {"id": 3388, "name": "space colony"}, {"id": 3679, "name": "society"}, {"id": 3801, "name": "space travel"}, {"id": 9685, "name": "futuristic"}, {"id": 9840, "name": "romance"}, {"id": 9882, "name": "space"}, {"id": 9951, "name": "alien"}, {"id": 10148, "name": "tribe"}, {"id": 10158, "name": "alien planet"}, {"id": 10987, "name": "cgi"}, {"id": 11399, "name": "marine"}, {"id": 13065, "name": "soldier"}, {"id": 14643, "name": "battle"}, {"id": 14720, "name": "love affair"}, {"id": 165431, "name": "anti war"}, {"id": 193554, "name": "power relations"}, {"id": 206690, "name": "mind and soul"}, {"id": 209714, "name": "3d"}]'

In [49]:
movies["cast"][0]

'[{"cast_id": 242, "character": "Jake Sully", "credit_id": "5602a8a7c3a3685532001c9a", "gender": 2, "id": 65731, "name": "Sam Worthington", "order": 0}, {"cast_id": 3, "character": "Neytiri", "credit_id": "52fe48009251416c750ac9cb", "gender": 1, "id": 8691, "name": "Zoe Saldana", "order": 1}, {"cast_id": 25, "character": "Dr. Grace Augustine", "credit_id": "52fe48009251416c750aca39", "gender": 1, "id": 10205, "name": "Sigourney Weaver", "order": 2}, {"cast_id": 4, "character": "Col. Quaritch", "credit_id": "52fe48009251416c750ac9cf", "gender": 2, "id": 32747, "name": "Stephen Lang", "order": 3}, {"cast_id": 5, "character": "Trudy Chacon", "credit_id": "52fe48009251416c750ac9d3", "gender": 1, "id": 17647, "name": "Michelle Rodriguez", "order": 4}, {"cast_id": 8, "character": "Selfridge", "credit_id": "52fe48009251416c750ac9e1", "gender": 2, "id": 1771, "name": "Giovanni Ribisi", "order": 5}, {"cast_id": 7, "character": "Norm Spellman", "credit_id": "52fe48009251416c750ac9dd", "gender": 

In [50]:
movies["crew"][0]

'[{"credit_id": "52fe48009251416c750aca23", "department": "Editing", "gender": 0, "id": 1721, "job": "Editor", "name": "Stephen E. Rivkin"}, {"credit_id": "539c47ecc3a36810e3001f87", "department": "Art", "gender": 2, "id": 496, "job": "Production Design", "name": "Rick Carter"}, {"credit_id": "54491c89c3a3680fb4001cf7", "department": "Sound", "gender": 0, "id": 900, "job": "Sound Designer", "name": "Christopher Boyes"}, {"credit_id": "54491cb70e0a267480001bd0", "department": "Sound", "gender": 0, "id": 900, "job": "Supervising Sound Editor", "name": "Christopher Boyes"}, {"credit_id": "539c4a4cc3a36810c9002101", "department": "Production", "gender": 1, "id": 1262, "job": "Casting", "name": "Mali Finn"}, {"credit_id": "5544ee3b925141499f0008fc", "department": "Sound", "gender": 2, "id": 1729, "job": "Original Music Composer", "name": "James Horner"}, {"credit_id": "52fe48009251416c750ac9c3", "department": "Directing", "gender": 2, "id": 2710, "job": "Director", "name": "James Cameron"},

In [51]:
def extract_names(obj):
    try:
        obj = ast.literal_eval(obj)
        return [i['name'] for i in obj]
    except:
        return []

def extract_top_cast(obj):
    try:
        obj = ast.literal_eval(obj)
        return [i['name'] for i in obj[:3]]
    except:
        return []

def extract_director(obj):
    try:
        obj = ast.literal_eval(obj)
        for i in obj:
            if i['job'] == 'Director':
                return [i['name']]
        return []
    except:
        return []

In [52]:
movies["genres"] = movies["genres"].apply(extract_names)
movies["keywords"] = movies["keywords"].apply(extract_names)
movies["cast"] = movies["cast"].apply(extract_top_cast)
movies["director"] = movies["crew"].apply(extract_director)

# combine everything
movies["tags"] = movies.apply(
    lambda x: " ".join(
        x["genres"] + x["keywords"] + x["cast"] + x["director"] + [str(x["overview"])]
    ),
    axis=1
)

movies = movies[["title", "tags"]]

In [53]:
movies.head()

,title,tags
0,Avatar,Action Adventure Fantasy Science Fiction cultu...
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action ocean drug abuse exot...
2,Spectre,Action Adventure Crime spy based on novel secr...
3,The Dark Knight Rises,Action Crime Drama Thriller dc comics crime fi...
4,John Carter,Action Adventure Science Fiction based on nove...


In [54]:
movies['tags'][0]

'Action Adventure Fantasy Science Fiction culture clash future space war space colony society space travel futuristic romance space alien tribe alien planet cgi marine soldier battle love affair anti war power relations mind and soul 3d Sam Worthington Zoe Saldana Sigourney Weaver James Cameron In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.'

In [55]:
movies.isnull().sum()

,0
title,0
tags,0


In [57]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

tfidf_matrix = tfidf.fit_transform(movies["tags"])

tfidf_sim = cosine_similarity(tfidf_matrix)

In [58]:
def recommend_tfidf(movie):
    idx = movies[movies["title"] == movie].index[0]

    scores = list(enumerate(tfidf_sim[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    top10 = scores[1:11]

    return [movies.iloc[i[0]].title for i in top10]

**USING INSTRUCT-MODEL EMBEDDINGS**

In [59]:
!pip install -q sentence-transformers

**We have used BAAI/bge-base-en-v1.5, which is a 110M parameter transformer-based sentence embedding model built on a BERT-base encoder architecture. It is trained using contrastive learning for retrieval tasks, making it well-suited for semantic similarity and recommendation systems.**

Architecture type: BERT-base (Transformer Encoder architecture)

1. Layers: 12

2. Hidden size: 768

3. Attention heads: 12

4. Pooling: Mean pooling (to get sentence embedding)






In [60]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("BAAI/bge-base-en-v1.5")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [61]:
def build_input(text):
    return "Represent this movie for retrieval: " + text

In [62]:
embeddings = model.encode(
    [build_input(t) for t in movies["tags"]],
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=64
)

Batches:   0%|          | 0/76 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [65]:
import os

save_path = f"{BASE_DIR}/tmdb-5000-embeddings-BAAI"

os.makedirs(save_path, exist_ok=True)

In [66]:
import numpy as np

np.save(f"{save_path}/embeddings.npy", embeddings)

NameError: name 'embeddings' is not defined

In [ ]:
print("Shape:", embeddings.shape)

print("\nFirst embedding (first 10 values):")
print(embeddings[0][:10])

print("\nFirst 3 embeddings (first 5 values each):")
print(embeddings[:3, :5])

In [67]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def recommend_bge(movie, embeddings_path):


    embeddings = np.load(embeddings_path)


    idx = movies[movies["title"] == movie].index[0]


    scores = cosine_similarity([embeddings[idx]], embeddings)[0]


    top10 = scores.argsort()[::-1][1:11]

    return [movies.iloc[i].title for i in top10]

In [68]:
path = f"{save_path}/embeddings.npy"

recommend_bge("Avatar", path)

['Battle: Los Angeles',
 'Terminator Genisys',
 'Starship Troopers',
 'Æon Flux',
 'Prometheus',
 'Supernova',
 'The Inhabited Island',
 'Aliens',
 'Damnation Alley',
 'Battlefield Earth']

In [69]:
recommend_tfidf("Avatar")

['Aliens',
 'Alien',
 'Moonraker',
 'Alien³',
 'Silent Running',
 'Spaceballs',
 'Mission to Mars',
 'Lost in Space',
 'Planet of the Apes',
 'Lifeforce']